# Automatic Differentiation

In that section, deriving the gradient $\nabla_{\mathbf{w}} L = \mathbf{X}^\top (2\mathbf{r}) = (2, 4)$ of our two-parameter line model by hand was straightforward, whereas doing so across dozens of layers and billions of parameters would be hopeless. Modern deep learning frameworks automate this via *automatic differentiation* (*autograd*): as a program evaluates a scalar loss, the framework traces the elementary operations into a *computational graph* and traverses that graph in reverse (*backpropagation*) to apply the chain rule mechanically. Remarkably, whether a model has two parameters or a billion, a single backward sweep computes the exact gradient with respect to *all* of them at only a small constant multiple of the forward-pass cost.

In [1]:
import jax
from jax import numpy as jnp

## Computing Gradients

### The Forward Pass

We first set up the line model $L(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$ at $\mathbf{w} = (1, 0)$ from that section using framework tensors, writing only the forward pass:

In [2]:
X = jnp.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])  # [1, input]
y = jnp.array([1.0, 1.0, 0.0])
w = jnp.array([1.0, 0.0])

def loss(w):
    return ((X @ w - y) ** 2).sum()

JAX takes a purely functional approach: arrays carry no mutable gradient buffers, so evaluating `loss(w)` on its own runs ordinary array arithmetic and returns `1.0`. Instead, `jax.grad(loss)` will trace `loss` when we differentiate it below:

In [3]:
loss(w)

Array(1., dtype=float32)

The recorded trace forms the *computational graph* shown in the figure, breaking the loss into four primitive operations: $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$, $\mathbf{r} = \hat{\mathbf{y}} - \mathbf{y}$, $\mathbf{s} = \mathbf{r} \odot \mathbf{r}$, and $L = \sum_i s_i$.

![The computational graph of $L = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$ at $\mathbf{w} = (1, 0)$, with one node for $\mathbf{w}$ and for each value computed from it; the constants $\mathbf{X}$ and $\mathbf{y}$ are not drawn, and $\mathbf{r} \odot \mathbf{r}$ is the elementwise square. Each node shows its value (black) and the gradient of $L$ with respect to it (red). The forward pass (blue) runs from left to right; the backward pass (red) starts from $1$ at $L$ and applies the local rule of each operation, from right to left.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-comp-graph.svg)

### The Backward Pass

Triggering the backward pass propagates sensitivities from $\partial L/\partial L = 1$ at the output back to $\mathbf{w}$ through each node's local derivative rule,

$$\nabla_{\mathbf{s}} L = \mathbf{1} \to \nabla_{\mathbf{r}} L = 2\mathbf{r} \to \nabla_{\hat{\mathbf{y}}} L = 2\mathbf{r} \to \nabla_{\mathbf{w}} L = \mathbf{X}^\top(2\mathbf{r}),$$

reproducing $(2, 4)$:

`jax.grad(loss)(w)` evaluates the gradient function at `w`.

In [4]:
jax.grad(loss)(w)

Array([2., 4.], dtype=float32)

Because local backward rules require intermediate activations from the forward pass (such as $\mathbf{r}$ and $\mathbf{X}$), training consumes substantially more memory than inference (the figure).

![Memory during a forward pass through four layers with outputs $\mathbf{h}_1, \ldots, \mathbf{h}_4$. (a) When gradients are tracked, every output is kept, because the backward pass (red) will need it. (b) When they are not, as during inference, an output can be freed as soon as the next layer has used it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-activation-memory.svg)

### Repeated Backward Passes

What happens when we evaluate the gradient a second time at the same parameter vector $\mathbf{w}$?

Because `jax.grad(loss)(w)` is a pure function that returns a fresh array on every call and stores no state on `w`, calling it repeatedly always returns `(2, 4)` without side effects; to accumulate gradients across micro-batches, you simply sum the returned arrays explicitly.

### Vector-Valued Functions

When an operation produces a vector of $m$ outputs $\mathbf{u} \in \mathbb{R}^m$ from $n$ inputs $\mathbf{x} \in \mathbb{R}^n$ with $m \times n$ Jacobian $\mathbf{J}$, a backward pass seeded with a weight vector $\mathbf{v} \in \mathbb{R}^m$ computes the transposed *vector--Jacobian product* $\mathbf{J}^\top \mathbf{v} = \nabla_{\mathbf{x}}(\mathbf{v}^\top \mathbf{u})$ without materializing $\mathbf{J}$. Passing $\mathbf{v} = \mathbf{1}$ on the per-example squared errors $(r_1^2, r_2^2, r_3^2)$ differentiates their sum:

`jax.vjp(losses, w)` returns the forward value and a function mapping $\mathbf{v}$ to $(\mathbf{J}^\top\mathbf{v},)$.

In [5]:
def losses(w):
    return (X @ w - y) ** 2  # one loss per example

_, vjp = jax.vjp(losses, w)  # vjp maps v to J^T v
vjp(jnp.ones(3))[0]  # v = 1

Array([2., 4.], dtype=float32)

## Controlling the Graph

### Detaching Values

*Detaching* a tensor creates a view or copy with the same numerical values but no link to its ancestors in the computational graph, so backpropagation treats it as a constant. Consider $u = x^2$ and $z = u x$ on $\mathbf{x} = (0, 1, 2, 3)$ (the figure): when $u$ is attached, gradient flows along both paths to give $\partial z/\partial x = 3x^2$; when $u$ is detached to $c$, differentiating $z = c x$ treats $c$ as a constant and gives $\partial z/\partial x = c = x^2$ (essential whenever a network computes its own bootstrap target, as in deep Q-learning, that section).

![The product $z = ux$ with $u = x^2$, (a) attached and (b) detached. Blue arrows are operations of the forward pass; red arrows show the backward pass, labeled with the local derivatives it multiplies by. Detaching $u$ creates a constant $c$ with the same value and no connection to $x$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-detach.svg)

`jax.lax.stop_gradient(u)` blocks gradients inside a differentiated function.

In [6]:
x = jnp.arange(4.0)

def z(x):
    u = x * x
    c = jax.lax.stop_gradient(u)  # values of u, no history
    return c * x

jax.grad(lambda x: z(x).sum())(x)

Array([0., 1., 4., 9.], dtype=float32)

By contrast, if we compute `z = u * x` directly without detaching `u`, backpropagation flows through both factors of $z = x^3$ and returns $\partial z / \partial x = 3x^2 = (0, 3, 12, 27)$:

In [7]:
jax.grad(lambda x: (x * x * x).sum())(x)

Array([ 0.,  3., 12., 27.], dtype=float32)

### Disabling Gradient Tracking

During validation, inference, or parameter updates $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$, we disable gradient tracking so those operations are not recorded on the loss graph:

Outside `jax.grad`, array arithmetic runs untraced.

In [8]:
w = w - 0.1 * jax.grad(loss)(w)
w, loss(w)

(Array([ 0.8, -0.4], dtype=float32), Array(0.4, dtype=float32))

As in that section, one step with $\eta = 0.1$ moves $\mathbf{w}$ from $(1, 0)$ to $(0.8, -0.4)$ and lowers the loss from $1$ to $0.4$.

## Differentiating Programs

### Control Flow

Because autograd records the sequence of elementary operations that *actually executed* on a given input, it differentiates Python functions containing data-dependent `while` loops and `if`/`else` branches directly:

In [9]:
def f(a):
    b = a * 2
    while jnp.linalg.norm(b) < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

For any nonzero scalar $a$ (at $a = 0$ the `while` loop would never terminate), `f(a)` performs a fixed sequence of multiplications by $2$ (and possibly $100$), so `f(a) = k * a` is piecewise linear with local slope $k = f(a)/a$ (the figure). Specifically, $a = 0.3$ doubles $12$ times to reach $0.3 \times 2^{12} = 1228.8 \ge 1000$ and takes the positive branch ($k = 2^{12} = 4096$), whereas $a = -0.75$ doubles $11$ times to $-1536$ and takes the `else` branch ($k = 100 \times 2^{11} = 204{,}800$). Autograd recovers both slopes:

In [10]:
for value in (0.3, -0.75):
    a = jnp.array(value)
    print(f'a = {value:5.2f}   f(a) / a = {f(a) / a:6.0f}   '
          f'gradient = {jax.grad(f)(a):6.0f}')

a =  0.30   f(a) / a =   4096   gradient =   4096
a = -0.75   f(a) / a = 204800   gradient = 204800


![The graphs recorded by `f` for two inputs. For $a = 0.3$, `f` doubles $a$ $12$ times and the first branch runs; for $a = -0.75$, it doubles $a$ $11$ times and the second branch runs. Each graph is linear in $a$, with its own slope.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-dynamic.svg)

Keep in mind that autograd differentiates the *executed trace* rather than the abstract mathematical function: at a threshold point where the loop count jumps, $f(a)$ is discontinuous, yet autograd returns the slope of whichever branch ran.

## Forward and Reverse Mode

For a mapping $\mathbf{f} : \mathbb{R}^n \to \mathbb{R}^m$ with $m \times n$ Jacobian $\mathbf{J}$, the chain rule can be traversed in either direction (the figure; see @Wengert.1964; @Linnainmaa.1970 and that section):

* **Reverse mode (vector--Jacobian product, VJP):** A reverse sweep seeded with $\mathbf{v} \in \mathbb{R}^m$ at the outputs computes $\mathbf{J}^\top \mathbf{v}$; seeding with the $j^\textrm{th}$ standard basis vector $\mathbf{v} = \mathbf{e}_j$ extracts row $j$ of $\mathbf{J}$ ($m$ sweeps for the full Jacobian).
* **Forward mode (Jacobian--vector product, JVP):** A forward sweep seeded with a tangent $\mathbf{d} \in \mathbb{R}^n$ at the inputs propagates derivatives forward alongside values to compute $\mathbf{J}\mathbf{d}$; seeding with $\mathbf{d} = \mathbf{e}_i$ extracts column $i$ of $\mathbf{J}$ ($n$ sweeps for the full Jacobian, with constant $2\times$ memory overhead).

![Two ways to fill the $m \times n$ Jacobian $\mathbf{J}$ of a function with $n$ inputs and $m$ outputs: (a) forward mode, $n$ sweeps in all; (b) reverse mode, $m$ sweeps in all.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-fwd-vs-rev.svg)

JAX provides `jax.vjp`, `jax.jvp`, `jax.jacrev`, and `jax.jacfwd`.

A scalar loss function has $m = 1$ output and $n$ parameters, so its Jacobian is a single row $\nabla_{\mathbf{w}} L^\top$. Reverse mode (*backpropagation*) computes all $n$ components of the gradient in *one sweep*, whereas forward mode would require $n$ sweeps, which explains why reverse mode is universal for training deep networks.

## Summary

Automatic differentiation traces the elementary operations of a forward pass into a computational graph and applies local derivative rules in reverse to compute exact gradients, reproducing $\nabla_{\mathbf{w}} L = (2, 4)$ on our line model without hand-written derivative code. Detaching an intermediate value severs its history so backpropagation treats it as a constant, while disabling gradient tracking keeps parameter updates and inference evaluations off the graph and avoids caching forward activations. Because the graph records the operations that actually executed on a given input, autograd differentiates seamlessly through Python loops and conditionals. Finally, one forward-mode sweep computes a Jacobian--vector product $\mathbf{J}\mathbf{d}$ (one column per basis seed) whereas one reverse-mode sweep computes a vector--Jacobian product $\mathbf{J}^\top\mathbf{v}$ (one row per basis seed), so for a scalar loss ($m = 1$) a single reverse sweep yields the entire $n$-parameter gradient.

## Exercises

1. [code] **Detached intermediates.** Write three functions of `x` that
   compute `u = x * x` and `c = jax.lax.stop_gradient(u)` and return
   `c * c`, `u * c`, and `u * u`, respectively. Before running any code,
   predict the gradient of the sum of the entries of each result with
   respect to `x`. Check your predictions with `jax.grad`, and explain the
   ratio between the results for `u * c` and `u * u`.

2. [code] **Manual parameter updates.** Let `w = jnp.arange(4.0)` and
   `g = lambda x: (w * x).sum()`, which reads `w` from the enclosing scope.
   Predict `jax.grad(g)(x)`, check, and state with respect to which array
   `jax.grad` differentiates. Then rewrite `g` so that `jax.grad` returns
   the gradient of `(w * x).sum()` with respect to `w`, use it for one
   update of `w` with learning rate 0.1, and explain what this implies for
   how a JAX training loop must pass the parameters to the loss function.

3. [code] **From scalar to vector input.** Replace the scalar input `a` of
   the function `f` defined in that section by a
   random vector of length 5, and differentiate the sum of the entries of
   `fa = f(a)` with respect to `a`.
    1. Express `f(a)` in terms of `a`, and give the Jacobian of `f` at `a`
       and the gradient of the sum of the entries of `fa`.
    1. Compute this gradient with automatic differentiation and check
       whether it still equals `fa / a` elementwise.
    1. Describe the set of inputs `a` at which `f` jumps. Which part of
       this set has no counterpart for a scalar input?

1. [code] **Programs versus functions.** Automatic differentiation
   differentiates the operations that a program executes, not the function
   that the program computes.
    1. The functions $|x|$ and $\sqrt{x^2}$ are equal. Compute the
       derivative of each at $x = 0$ by automatic differentiation, using the
       framework's absolute value and square root. Explain both results from
       the local derivatives of the operations involved. Neither function
       has a derivative at $0$; which of the two results does more harm in
       training, and why?
    1. Write a program that returns $x^2$ for every input $x$ but whose
       automatic derivative at $x = 0$ is $1$ instead of $0$. Explain why
       automatic differentiation cannot detect the difference.

1. **Cost of second derivatives.** ● The counting argument in
   that section puts the cost of the gradient of a
   scalar loss with $n$ parameters at one reverse sweep. Use the same argument to
   estimate, in units of function evaluations, the cost of the full Hessian
   (the $n \times n$ matrix of second derivatives) and the cost of a single
   Hessian–vector product. Explain the difference between the two costs.

[Discussions](https://d2l.discourse.group/t/17970)